In [5]:
!python -m data_gen.generate_meta_params

Generating Stabilized Meta-Parameters...
Saved stabilized meta-params to data/meta_params.npz
Generated: 100 Train, 20 Val
Generated 20 for TestA
Generated 20 for TestB
Generated 20 for TestC


In [6]:
!python -m data_gen.generate_trajectories

Starting Data Generation on device: cuda
Processing split: train (100 thetas)
Simulating train: 100%|███████████████████████| 100/100 [00:36<00:00,  2.77it/s]
Processing split: val (20 thetas)
Simulating val: 100%|███████████████████████████| 20/20 [00:03<00:00,  5.87it/s]
Processing split: testA (20 thetas)
Simulating testA: 100%|█████████████████████████| 20/20 [00:06<00:00,  2.99it/s]
Processing split: testB (20 thetas)
Simulating testB: 100%|█████████████████████████| 20/20 [00:06<00:00,  3.04it/s]
Processing split: testC (20 thetas)
Simulating testC: 100%|█████████████████████████| 20/20 [00:04<00:00,  4.80it/s]
Done! Generated 11160 trajectories. Index saved to data/index.csv


generated 11,160 valid trajectories
Train: 100 $\theta$ × (64 + 16 trajectories) = 8,000
Val: 20 $\theta$ × 32 trajectories = 640
Test A: 20 $\theta$ × (10 + 32 trajectories) = 840
Test B: 20 $\theta$ × (10 + 32 trajectories) = 840
Test C: 20 $\theta$ × (10 + 32 trajectories) = 840
Total: 11,160 files.

sanity for loaders/trajectory_datasets.py

In [8]:
import torch
from torch.utils.data import DataLoader
from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset

print(f"🔍 Testing Dataloader on device: {cfg.device}")
print(f"   Expected Sequence Length: {cfg.time_grid.n_steps + 1}")
print(f"   Expected Dimension: {cfg.basis.x_dim}\n")

# --- Test 1: Load Training Data (Inner Loop) ---
try:
    train_ds = TrajectoryDataset(
        index_path=cfg.paths.data_root + "index.csv",
        split="train",
        role="train_inner"
    )
    print(f"✅ Loaded Train Dataset: {len(train_ds)} items (Expected ~6400)")
    
    # Check Batch Loading
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    batch_x, batch_ids = next(iter(train_loader))
    
    print(f"   Batch Shape: {batch_x.shape}")  # Should be (4, 201, 5)
    print(f"   Sample IDs: {batch_ids}")
    
    # Verify exact shape matches config
    T, d = batch_x.shape[1], batch_x.shape[2]
    assert T == cfg.time_grid.n_steps + 1, f"Time dimension mismatch! Got {T}"
    assert d == cfg.basis.x_dim, f"State dimension mismatch! Got {d}"
    print("   Shape check passed.\n")

except Exception as e:
    print(f"❌ Train Dataset Failed: {e}\n")

# --- Test 2: Load Test C (Support Set) ---
# This verifies your 'Test C' fix generated readable data
try:
    test_c_ds = TrajectoryDataset(
        index_path=cfg.paths.data_root + "index.csv",
        split="testC",
        role="support"
    )
    print(f"✅ Loaded Test C (Support): {len(test_c_ds)} items (Expected ~200)")
    
    # Check one item
    x, theta_id = test_c_ds[0]
    print(f"   Item 0 Shape: {x.shape}")
    print(f"   Item 0 ID: {theta_id}\n")

except Exception as e:
    print(f"❌ Test C Dataset Failed: {e}\n")

# --- Test 3: Intentional Failure (Safety Check) ---
# This ensures your 'RuntimeError' logic works for empty filters
print("running safety check...")
try:
    # There is no role 'garbage_role'
    bad_ds = TrajectoryDataset(
        index_path=cfg.paths.data_root + "index.csv",
        split="train",
        role="garbage_role"
    )
    print("❌ FAILED: Should have raised RuntimeError for empty dataset!")
except RuntimeError as e:
    print(f"✅ Safety Check Passed: Correctly caught empty dataset error.\n   (Error msg: {e})")
except Exception as e:
    print(f"❌ FAILED: Raised wrong error type: {type(e)}")

print("\n🎉 Dataloader Verification Complete!")

🔍 Testing Dataloader on device: cuda
   Expected Sequence Length: 201
   Expected Dimension: 5

✅ Loaded Train Dataset: 6400 items (Expected ~6400)
   Batch Shape: torch.Size([4, 201, 5])
   Sample IDs: ('train_046', 'train_062', 'train_079', 'train_065')
   Shape check passed.

✅ Loaded Test C (Support): 200 items (Expected ~200)
   Item 0 Shape: torch.Size([201, 5])
   Item 0 ID: testC_000

running safety check...
✅ Safety Check Passed: Correctly caught empty dataset error.
   (Error msg: No rows found in index for split='train', role='garbage_role'. Check your index.csv or generation parameters.)

🎉 Dataloader Verification Complete!


models/mlp.py sanity check 

In [1]:
import torch
from models.mlp import MLP

# --- Test Parameters ---
input_dim = 10
hidden_dims = [32, 16]
output_dim = 5
batch_size = 4

print(f"🔍 Testing MLP on device: CPU")

try:
    # 1. Instantiate the model
    # We test with LayerNorm enabled to be sure that logic works too
    model = MLP(
        input_dim=input_dim, 
        hidden_dims=hidden_dims, 
        output_dim=output_dim, 
        use_layernorm=True
    )
    print("✅ MLP Instantiated successfully.")
    print(f"   Structure: {model}")

    # 2. Create dummy input
    x = torch.randn(batch_size, input_dim)
    
    # 3. Forward Pass
    y = model(x)
    print(f"   Input shape:  {x.shape}")
    print(f"   Output shape: {y.shape}")

    # 4. Verify Shape
    expected_shape = (batch_size, output_dim)
    assert y.shape == expected_shape, f"Shape Mismatch! Expected {expected_shape}, got {y.shape}"
    print("✅ Shape check passed.")

    print("\n🎉 MLP Verification Complete!")

except Exception as e:
    print(f"\n❌ MLP Failed: {e}")

🔍 Testing MLP on device: CPU
✅ MLP Instantiated successfully.
   Structure: MLP(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=32, bias=True)
    (1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (2): SiLU()
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
    (5): SiLU()
    (6): Linear(in_features=16, out_features=5, bias=True)
  )
)
   Input shape:  torch.Size([4, 10])
   Output shape: torch.Size([4, 5])
✅ Shape check passed.

🎉 MLP Verification Complete!


sanity for models/encoder.py

In [4]:
import torch
from models.encoder import TrajEncoder

print(f"🔍 Testing Deep Encoder...")

try:
    # 1. Instantiate (Deep version)
    encoder = TrajEncoder(x_dim=5, z_dim=8, hidden_dim=32, num_layers=2, dropout=0.1)
    print("✅ Deep Encoder Instantiated.")
    
    # 2. Valid Input
    dummy_traj = torch.randn(4, 50, 5) # (Batch, Seq, Dim)
    z = encoder(dummy_traj)
    print(f"   Output shape: {z.shape}")
    assert z.shape == (4, 8)
    print("✅ Forward pass successful.")

    # 3. Test Safety Check
    print("   Testing Safety Guard...")
    bad_input = torch.randn(50, 5) # Missing batch dim
    try:
        encoder(bad_input)
        print("❌ FAILED: Should have caught bad dimension!")
    except ValueError as e:
        print(f"✅ Safety Guard Passed. Caught error: {e}")

    print("\n🎉 Encoder Verification Complete!")

except Exception as e:
    print(f"\n❌ Encoder Test Failed: {e}")

🔍 Testing Deep Encoder...
✅ Deep Encoder Instantiated.
   Output shape: torch.Size([4, 8])
✅ Forward pass successful.
   Testing Safety Guard...
✅ Safety Guard Passed. Caught error: Encoder input must be (batch, seq, x_dim), got torch.Size([50, 5])

🎉 Encoder Verification Complete!


safety for neural_sde.py

In [7]:
# sanity_neural_sde.py

import torch
from models.neural_sde import NeuralSDE

def main():
    print("🔍 Testing NeuralSDE...\n")

    # Parameters
    batch_size = 4
    x_dim = 5
    z_dim = 8
    hidden_dim = 32

    try:
        # 1. Instantiate model
        sde = NeuralSDE(x_dim=x_dim, z_dim=z_dim, hidden_dim=hidden_dim)
        print("✅ NeuralSDE instantiated.")

        # 2. Dummy inputs
        t = torch.tensor(0.5)                    # scalar time (unused, but required by API)
        y = torch.randn(batch_size, x_dim)       # state
        z = torch.randn(batch_size, z_dim)       # context

        # 3. Test drift f(t, y, z)
        drift = sde.f(t, y, z)
        print(f"   Drift shape: {drift.shape}")
        assert drift.shape == (batch_size, x_dim)
        print("✅ Drift calculation successful.")

        # 4. Test diffusion g(t, y, z)
        diffusion = sde.g(t, y, z)
        print(f"   Diffusion shape: {diffusion.shape}")
        assert diffusion.shape == (batch_size, x_dim, x_dim)

        # Only check positivity on the diagonal (off-diagonals are zero by design)
        diag = diffusion.diagonal(dim1=-2, dim2=-1)  # (batch, x_dim)
        min_diag = diag.min().item()
        print(f"   Min diagonal diffusion value: {min_diag:.6f}")
        if min_diag <= 0:
            raise ValueError("Diagonal diffusion entries must be strictly positive!")
        print("✅ Diffusion positivity check (diagonal) passed.")

        # 5. Optional: test broadcasting when z is a single vector
        z_single = torch.randn(z_dim)  # (z_dim,)
        drift_single = sde.f(t, y, z_single)
        diffusion_single = sde.g(t, y, z_single)
        assert drift_single.shape == (batch_size, x_dim)
        assert diffusion_single.shape == (batch_size, x_dim, x_dim)
        print("✅ z broadcasting (single vector) works as expected.")

        print("\n🎉 NeuralSDE verification complete. All checks passed.")

    except Exception as e:
        print(f"\n❌ NeuralSDE test failed: {e}")

if __name__ == "__main__":
    main()


🔍 Testing NeuralSDE...

✅ NeuralSDE instantiated.
   Drift shape: torch.Size([4, 5])
✅ Drift calculation successful.
   Diffusion shape: torch.Size([4, 5, 5])
   Min diagonal diffusion value: 0.344959
✅ Diffusion positivity check (diagonal) passed.
✅ z broadcasting (single vector) works as expected.

🎉 NeuralSDE verification complete. All checks passed.


safety for head.py

In [8]:
import torch
from models.head import ForecastHead

print(f"🔍 Testing ForecastHead...")

try:
    # 1. Instantiate
    head = ForecastHead(x_dim=5, z_dim=8, hidden_dim=32)
    print("✅ ForecastHead Instantiated.")

    # 2. Dummy Data
    x = torch.randn(4, 5) # Batch of states
    z = torch.randn(4, 8) # Batch of contexts

    # 3. Forward Pass
    pred = head(x, z)
    
    # 4. Check Shape
    print(f"   Input x: {x.shape}")
    print(f"   Output:  {pred.shape}")
    assert pred.shape == (4, 5)
    print("✅ Forward pass successful.")

    print("\n🎉 All Models Verified!")

except Exception as e:
    print(f"\n❌ Head Test Failed: {e}")

🔍 Testing ForecastHead...
✅ ForecastHead Instantiated.
   Input x: torch.Size([4, 5])
   Output:  torch.Size([4, 5])
✅ Forward pass successful.

🎉 All Models Verified!
